# Agente Explicador de Regras de Futebol com RAG (LangChain)

## Objetivo

Construir um agente conversacional simples utilizando `Retrieval-Augmented Generation (RAG)` para responder perguntas sobre *regras do futebol*, com base em *documentos oficiais*.

## Conceitos Abordados

* Definição do problema;
* Seleção da base de conhecimento;
* Preparação dos documentos;
* Embeddings e banco vetorial;
* Recuperação de contexto (Retriever);
* Integração com LLM;
* Testes e validação das respostas.

In [1]:
import os

# Loader de documentos PDF
from langchain_community.document_loaders import PyPDFLoader

# Divisão de texto em blocos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings do Google Generative AI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Banco Vetorial
from langchain_community.vectorstores import Chroma

# LLM
from langchain_google_genai import ChatGoogleGenerativeAI

# Cadeia RAG
from langchain_classic.chains import RetrievalQA

from dotenv import load_dotenv

from pydantic import SecretStr

from google import genai

/var/folders/4r/zr0k1g517dd2802bsyl_q6zh0000gn/T/ipykernel_15677/1025046631.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### 1. Definição do Problema

LLM's possuem conhecimento estático e podem alucinar. O objetivo aqui é garantir **respostas confiáveis**, conectando o modelo a documentos oficiais sobre regras de futebol.

In [2]:
# Caminho para o arquivo PDF com as regras de futebol
pdf_path = "assets/Livro_de_Regras_2026_2027.pdf"

# Carregar o PDF
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Quantidade de páginas carregadas
len(documents)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/UFOEGA+MyriadPro-Regular', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(1850, 0, 4405548112), '/LastChar': 150, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [212, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 307, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 612, 0, 580, 0, 492, 0, 0, 652, 239, 0, 542, 472, 0, 658, 689, 532, 0, 538, 493, 497, 0, 558, 846, 0, 541, 0, 0, 0, 0, 0, 0, 0, 482, 0, 0, 0, 501, 0, 559, 0, 234, 0, 0, 236, 0, 555, 0, 0, 0, 0, 0, 0, 0, 481, 736, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/UFOEGA+MyriadPro-Regular', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontD

135

In [3]:
# Exibir os documentos carregados do PDF
documents

[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.4 (Windows)', 'creationdate': '2026-06-11T12:05:28+02:00', 'moddate': '2026-06-11T12:54:48-03:00', 'trapped': '/False', 'source': 'assets/Livro_de_Regras_2026_2027.pdf', 'total_pages': 135, 'page': 0, 'page_label': '1'}, page_content='1\nRegras \ndo Jogo\n26/27'),
 Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.4 (Windows)', 'creationdate': '2026-06-11T12:05:28+02:00', 'moddate': '2026-06-11T12:54:48-03:00', 'trapped': '/False', 'source': 'assets/Livro_de_Regras_2026_2027.pdf', 'total_pages': 135, 'page': 1, 'page_label': '2'}, page_content='32'),
 Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.4 (Windows)', 'creationdate': '2026-06-11T12:05:28+02:00', 'moddate': '2026-06-11T12:54:48-03:00', 'trapped': '/False', 'source': 'assets/Livro_de_Regras_2026_2027.pdf', 'total_pages': 135, 'page': 2, 'page_label': '3'}, page_cont

### 2. Preparação dos Documentos

Os documentos precisam ser divididos em pequenos blocos para facilitar a recuperação de contexto.

In [4]:
# Divide os documentos em chunks menores

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
len(chunks)

725

In [5]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

for model in client.models.list():
    if "embedContent" in (model.supported_actions or []):
        print(model.name)

# Inicializa Embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    api_key=SecretStr(api_key) if api_key else None
)

# Cria o banco vetorial Chroma em lotes, respeitando o limite do plano
# gratuito do Gemini (100 requisições de embedding por minuto)
import time

TAMANHO_LOTE = 90
PAUSA_SEGUNDOS = 61

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="./chroma_regras_futebol_db"
)

# IDs fixos: se a célula for executada de novo, os chunks são sobrescritos (upsert)
# em vez de duplicados
ids = [f"chunk-{i}" for i in range(len(chunks))]

for inicio in range(0, len(chunks), TAMANHO_LOTE):
    fim = min(inicio + TAMANHO_LOTE, len(chunks))
    vector_store.add_documents(chunks[inicio:fim], ids=ids[inicio:fim])
    print(f"Chunks {inicio + 1}-{fim} de {len(chunks)} adicionados")
    if fim < len(chunks):
        time.sleep(PAUSA_SEGUNDOS)

print("Banco vetorial criado com", vector_store._collection.count(), "chunks")

models/gemini-embedding-001
models/gemini-embedding-2


/var/folders/4r/zr0k1g517dd2802bsyl_q6zh0000gn/T/ipykernel_15677/2823372543.py:24: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


Chunks 1-90 de 725 adicionados
Chunks 91-180 de 725 adicionados
Chunks 181-270 de 725 adicionados
Chunks 271-360 de 725 adicionados
Chunks 361-450 de 725 adicionados
Chunks 451-540 de 725 adicionados
Chunks 541-630 de 725 adicionados
Chunks 631-720 de 725 adicionados
Chunks 721-725 de 725 adicionados
Banco vetorial criado com 725 chunks
